<a href="https://colab.research.google.com/github/Alphaz-006/boilerplate-rock-paper-scissors/blob/main/Image%20Classification%20with%20Transfer%20Learning%20(CIFAR-10%20%2B%20ResNet)/Image_Classification_with_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Image Classification with Transfer Learning

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input


=== Load data ===

In [9]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

 Resize CIFAR-10 (32x32) to 96x96 for ResNet

In [3]:
def preprocess(X, y):
    X = tf.image.resize(X, (96,96))
    X = preprocess_input(X)
    return X, y

In [4]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(64).map(preprocess)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64).map(preprocess)


=== Base model ===


In [5]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(96,96,3))
base_model.trainable = False  # freeze for transfer learning

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [10]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, validation_data=test_ds, epochs=10)


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1710s 2s/step - accuracy: 0.7953 - loss: 0.6225 - val_accuracy: 0.8448 - val_loss: 0.4492
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1651s 2s/step - accuracy: 0.8471 - loss: 0.4460 - val_accuracy: 0.8560 - val_loss: 0.4216
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1747s 2s/step - accuracy: 0.8618 - loss: 0.3958 - val_accuracy: 0.8619 - val_loss: 0.4129
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1702s 2s/step - accuracy: 0.8759 - loss: 0.3562 - val_accuracy: 0.8575 - val_loss: 0.4210
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1667s 2s/step - accuracy: 0.8854 - loss: 0.3233 - val_accuracy: 0.8661 - val_loss: 0.4125
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1745s 2s/step - accuracy: 0.8910 - loss: 0.3050 - val_accuracy: 0.8623 - val_loss: 0.4303
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1714s 2s/step - accuracy: 0.8997 - loss: 0.2774 - val_accuracy: 0.8648 - val_loss: 0.4298
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1668s 2s/step - accuracy: 0.9061 - loss: 0.2638 - 

In [11]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [12]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_fine = model.fit(train_ds, validation_data=test_ds, epochs=5)


Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2574s 3s/step - accuracy: 0.9079 - loss: 0.2542 - val_accuracy: 0.8723 - val_loss: 0.4298
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2640s 3s/step - accuracy: 0.9426 - loss: 0.1618 - val_accuracy: 0.8774 - val_loss: 0.4276
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2540s 3s/step - accuracy: 0.9591 - loss: 0.1188 - val_accuracy: 0.8817 - val_loss: 0.4361
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2580s 3s/step - accuracy: 0.9706 - loss: 0.0903 - val_accuracy: 0.8822 - val_loss: 0.4451
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2577s 3s/step - accuracy: 0.9787 - loss: 0.0688 - val_accuracy: 0.8845 - val_loss: 0.4603


In [13]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Final Test Accuracy: {test_acc:.2%}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 274s 2s/step - accuracy: 0.8845 - loss: 0.4603
Final Test Accuracy: 88.45%


In [14]:
model.save("cifar10_resnet_model.h5")